# Tutorial: Real Estate Investment Advisor — EDA, Modeling, and Streamlit

**Audience:** Data science learners building a small end-to-end machine learning project.

**Prerequisites:** Basic Python, pandas, and scikit-learn. From the project root, install `requirements-dev.txt` before running the notebook: `python -m pip install -r requirements-dev.txt`.

**Learning goals:**
- Inspect and parse a public property-listings dataset.
- Explore asking prices and location differences.
- Train the project’s current-price and investment-proxy models.
- Save artifacts that the Streamlit app loads.

The notebook and app share the project code under src/. The notebook generates the EDA and model artifacts; app.py reads those artifacts to serve interactive predictions.


## Outline

1. Load the data and inspect its fields.
2. Parse amounts, rates, and area values.
3. Explore listing prices by location.
4. Train and evaluate the models.
5. Review the artifacts consumed by Streamlit.
6. Run the app.
7. Map the outputs to business workflows.
8. Complete an exercise.

**Important target definitions:** Amount(in rupees) contains total asking amounts such as “42 Lac” and “1.40 Cr”. Price (in rupees) is treated as the per-square-foot rate. The dataset has no five-year outcomes, so the app’s five-year value is an adjustable growth scenario, not a supervised forecast.


In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display

# Find the project root whether Jupyter starts here or in notebooks/.
candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((path for path in candidates if (path / "app.py").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Start Jupyter from the project folder or its notebooks folder.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "India_Property_Listings.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

from src.features import parse_amount, parse_price, prepare_features
from src.eda import run as generate_eda
from src.train import train

print("Project:", PROJECT_ROOT)
print("Dataset size:", f"{DATA_PATH.stat().st_size / (1024 * 1024):.1f} MB")


## 1. Inspect the dataset

Start with a small preview so the notebook stays responsive. The training function later reads the full file (or a reproducible row sample you choose).


In [ ]:
preview_columns = [
    "Title",
    "Amount(in rupees)",
    "Price (in rupees)",
    "location",
    "Carpet Area",
    "Super Area",
    "Status",
    "Transaction",
    "Furnishing",
    "facing",
    "overlooking",
    "Bathroom",
    "Balcony",
    "Floor",
    "Car Parking",
    "Ownership",
]
preview = pd.read_csv(
    DATA_PATH,
    usecols=preview_columns,
    nrows=10_000,
    low_memory=False,
)
print("Preview rows:", f"{len(preview):,}")
print("Locations in preview:", preview["location"].nunique())
display(preview.head(5))


## 2. Parse the price fields

The listing file stores total asking amount and per-area rate in separate columns. Parsing them separately avoids training a total-price model on a rate field by mistake. Area and property features use the shared parsing code that the Streamlit app also uses.


In [ ]:
preview["asking_amount_rupees"] = preview["Amount(in rupees)"].map(parse_amount)
preview["source_rate_rupees_per_sqft"] = preview["Price (in rupees)"].map(parse_price)
preview_features = prepare_features(preview)
preview["carpet_area_sqft"] = preview_features["Carpet_Area_SqFt"]

# Fill missing source rates from total amount / carpet area for descriptive EDA only.
derived_rate = preview["asking_amount_rupees"] / preview["carpet_area_sqft"]
preview["rate_rupees_per_sqft"] = preview["source_rate_rupees_per_sqft"].where(
    preview["source_rate_rupees_per_sqft"].gt(0), derived_rate
)

summary = pd.DataFrame(
    {
        "value": [
            len(preview),
            preview["location"].nunique(),
            preview["asking_amount_rupees"].notna().sum(),
            preview["carpet_area_sqft"].notna().sum(),
        ]
    },
    index=[
        "Preview rows",
        "Distinct locations",
        "Rows with parsed total asking amount",
        "Rows with parsed carpet area",
    ],
)
display(summary)
display(
    preview[
        [
            "Amount(in rupees)",
            "asking_amount_rupees",
            "Price (in rupees)",
            "rate_rupees_per_sqft",
            "Carpet Area",
            "carpet_area_sqft",
        ]
    ].head(8)
)


## 3. Explore location differences

Compare medians rather than only averages because property asking prices are skewed and can contain unusually expensive listings. This chart uses the 10,000-row preview; the full EDA script writes a location table from all usable rows.


In [ ]:
usable = preview.loc[
    preview["asking_amount_rupees"].gt(0)
    & preview["carpet_area_sqft"].gt(0)
    & preview["rate_rupees_per_sqft"].gt(0)
].copy()

location_preview = (
    usable.groupby("location", dropna=False)
    .agg(
        listings=("asking_amount_rupees", "size"),
        median_asking_amount=("asking_amount_rupees", "median"),
        median_rate_per_sqft=("rate_rupees_per_sqft", "median"),
    )
    .reset_index()
)

top_locations = location_preview.nlargest(15, "median_asking_amount")
fig = px.bar(
    top_locations.sort_values("median_asking_amount"),
    x="median_asking_amount",
    y="location",
    color="listings",
    orientation="h",
    labels={
        "median_asking_amount": "Median total asking amount (INR)",
        "location": "Location",
        "listings": "Listings in preview",
    },
    title="Locations with the highest median asking amount",
)
fig.show()

display(location_preview.sort_values("median_rate_per_sqft", ascending=False).head(10))


## 4. Generate the full EDA report

This calls the same EDA script included in the project. It writes reports/eda_summary.md and reports/location_summary.csv for use in the app or project write-up.


In [ ]:
generate_eda(DATA_PATH)
location_summary = pd.read_csv(PROJECT_ROOT / "reports" / "location_summary.csv")
display(location_summary.head(10))


## 5. Train and evaluate the models

The first run defaults to a reproducible 50,000-row sample, matching the delivered model artifacts. Change MAX_ROWS to None to train on every usable listing; that full-data run takes longer.

The regression model predicts total current asking amount. The classifier predicts a constructed proxy: asking price per carpet-area square foot at or below the location’s training-set median. It does not predict realized investment returns.


In [ ]:
MAX_ROWS = 50_000  # Use None for the full dataset.
metrics = train(DATA_PATH, max_rows=MAX_ROWS, skip_mlflow=False)

metrics_table = pd.DataFrame(
    [
        {
            "Model": "Current total asking amount",
            "Rows": metrics["rows_used"],
            "MAE (INR)": metrics["regression"]["mae_rupees"],
            "RMSE (INR)": metrics["regression"]["rmse_rupees"],
            "R²": metrics["regression"]["r2"],
        },
        {
            "Model": "Below-local-median proxy",
            "Rows": metrics["classifier_rows"],
            "MAE (INR)": np.nan,
            "RMSE (INR)": np.nan,
            "R²": np.nan,
        },
    ]
)
display(metrics_table)
display(pd.Series(metrics["classification_proxy"], name="Proxy classifier metrics"))


## 6. Confirm the Streamlit artifacts

Training saves the fitted pipelines, benchmark thresholds, metrics, and feature importances in models/. The Streamlit app loads these files, so rerunning the notebook refreshes the models used by the app.


In [ ]:
import joblib

regressor = joblib.load(PROJECT_ROOT / "models" / "price_regressor.joblib")
classifier = joblib.load(PROJECT_ROOT / "models" / "investment_proxy_classifier.joblib")
benchmarks = json.loads(
    (PROJECT_ROOT / "models" / "benchmark_thresholds.json").read_text(encoding="utf-8")
)

sample_features = prepare_features(preview.iloc[[0]])
predicted_total = float(np.expm1(regressor.predict(sample_features)[0]))
proxy_probability = float(classifier.predict_proba(sample_features)[0, 1])

print(f"Example estimated current total asking amount: INR {predicted_total:,.0f}")
print(f"Classifier probability for the below-median proxy: {proxy_probability:.1%}")
print("Saved location benchmarks:", len(benchmarks["by_location_rupees_per_sqft"]))


## 7. Run the Streamlit app

From the project root, run this command in a terminal:

    python -m streamlit run app.py

On Windows, you can also run run_app.bat. The app reads the models created above and provides a property form, a local price-per-area comparison, EDA charts, and model metrics.

The five-year value is calculated as current value × (1 + selected annual rate)^years. The rate is an assumption the user can change; the data contains no five-year target.


## 8. Business use cases and decision improvement

This prototype can support these human-led workflows:

- **Buyer and agent shortlists:** a consistent per-area comparison can reduce manual arithmetic and prioritize comparable-property checks.
- **Marketplace listing review:** a shared location benchmark can help reviewers spot unusual seller prices earlier; keep seller choice and human override.
- **Brokerage sourcing:** a repeatable below-median screen can help analysts prioritize due diligence. The current form assesses one property at a time.
- **Regional market planning:** listing counts and asking-price levels can provide a consistent starting point for local research and marketing allocation.

A practical decision loop is to compare consistently, prioritize human review, make growth assumptions visible, and verify with recent closed sales and property details. These are expected benefits to evaluate, not proven gains. Compare review time, reviewer consistency, and flagged-case outcomes with the current workflow. The below-median label is not profitability, and the five-year value is an adjustable scenario because the dataset has no future-price outcomes.


## Exercise

Using the full location summary created in Step 4, find locations with at least 100 usable listings and show the ten lowest median rates per square foot. Which locations would be most useful to compare in the app? Remember that a low rate alone does not prove an investment will appreciate.


In [ ]:
location_summary = pd.read_csv(PROJECT_ROOT / "reports" / "location_summary.csv")
eligible_locations = location_summary.loc[location_summary["listings"] >= 100]
display(
    eligible_locations
    .sort_values("median_price_per_sqft")
    .head(10)
)


## Pitfalls and extensions

- Do not confuse the total asking amount with the per-square-foot rate.
- Do not describe the proxy label as true profitability or investment return.
- Do not describe the five-year scenario as a learned forecast; no future outcomes are present.
- Missing carpet areas reduce the rows available for price-per-area comparisons.
- Optional extension: compare model performance across locations and evaluate with a location-based split instead of a random split.
